In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_product_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ProductId  string "
        #     ",SourceSystem string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",ProductKey  string "
        #     ",ProductCode  string "
        #     ",ProductDescription  string"
        #     ",SubProductCode  int"
        #     ",SubProduct  string"
        #     ",BankingUnitCode  string"
        #     ",InterestUnit  string"
        #     ",InterestFrequency  int"
        #     ",DeliquentInterestUnit  string"
        #     ",DeliquentInterestFrequency  int"
        #     ",InterestDate  date"
        #     ",LastValeurDate  date"
        #     ",NextValeurDate  date"
        #     ",PeriodicIntFlag  string"
        #     ",ShadowFlag  boolean"
        #     ",SyndicatedFlag  boolean"
        #     ",L3869Flag  boolean"
        #     ",OCWFlag  boolean"
        #     ",BondedFlag  boolean"
        #     ",ProductCategory  string"
        #     ",BusinessDate  date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Product"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")


        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ========================== dim_subproduct ==========================
        dim_subproduct_extra_filters = None

        if dim_subproduct_extra_filters is not None:
            dim_subproduct_extra_filters.append(date_filter)
            dim_subproduct_final_filters = " AND ".join(dim_subproduct_extra_filters)
        else:
            dim_subproduct_final_filters = date_filter

        dim_subproduct_selected_fields = [
            "sprd_id",
            "sprd_desc",
            "SPRDH1_ID",
            "sprd_intcode",
            "businessdate",
        ]

        dim_subproduct = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_subproduct",
            dim_subproduct_final_filters,
            dim_subproduct_selected_fields,
        )

        # ================== dim_subproduct_hier1 ==================

        dim_subproduct_hier1_extra_filters = None

        if dim_subproduct_hier1_extra_filters is not None:
            dim_subproduct_hier1_extra_filters.append(date_filter)
            dim_subproduct_hier1_final_filters = " AND ".join(
                dim_subproduct_hier1_extra_filters
            )
        else:
            dim_subproduct_hier1_final_filters = date_filter

        dim_subproduct_hier1_selected_fields = ["SPRDH1_ID", "SPRDH1_DESC"]

        dim_subproduct_hier1 = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_subproduct_hier1",
            dim_subproduct_hier1_final_filters,
            dim_subproduct_hier1_selected_fields,
        )

        # ================== Join ==================
        dim_subproduct_status_dim_subproduct_hier1_status_join_condition = (
            dim_subproduct["SPRDH1_ID"] == dim_subproduct_hier1["SPRDH1_ID"]
        )

        product_qca = dim_subproduct.join(
            dim_subproduct_hier1,
            dim_subproduct_status_dim_subproduct_hier1_status_join_condition,
            "left",
        )

        dim_subproduct_primary_key_fields = [dim_subproduct.sprd_id, lit("qca")]

        product_qca = product_qca.withColumn(
            "ProductId",
            sha2(concat_ws(";", *dim_subproduct_primary_key_fields), 256),
        ).withColumn(
            "ProductCode",
            dim_subproduct.sprd_id.cast("string"),
        )

        product_qca = product_qca.select(
            product_qca.ProductId,
            lit("qca").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            product_qca.ProductCode.cast("string").alias("ProductKey"),
            product_qca.ProductCode.cast("string"),
            product_qca.sprd_desc.cast("string").alias("ProductDescription"),
            dim_subproduct_hier1.SPRDH1_ID.cast("int").alias("SubProductCode"),
            dim_subproduct_hier1.SPRDH1_DESC.cast("string").alias("SubProduct"),
            product_qca.sprd_intcode.cast("string").alias("BankingUnitCode"),
            lit(None).cast("string").alias("InterestUnit"),
            lit(None).cast("int").alias("InterestFrequency"),
            lit(None).cast("string").alias("DeliquentInterestUnit"),
            lit(None).cast("int").alias("DeliquentInterestFrequency"),
            lit(None).cast("date").alias("InterestDate"),
            lit(None).cast("date").alias("LastValeurDate"),
            lit(None).cast("date").alias("NextValeurDate"),
            lit(None).cast("string").alias("PeriodicIntFlag"),
            lit(None).cast("boolean").alias("ShadowFlag"),
            lit(None).cast("boolean").alias("SyndicatedFlag"),
            lit(None).cast("boolean").alias("L3869Flag"),
            lit(None).cast("boolean").alias("OCWFlag"),
            lit(None).cast("boolean").alias("BondedFlag"),
            lit(None).cast("string").alias("ProductCategory"),
            product_qca.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        product_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_product_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)